In [1]:
import ast
import os

In [2]:
from urllib.parse import urlparse

def get_filename(url: str):
    if url[-4:] == ".git":
        url = url[:-4]
        
    parts = urlparse(url).path.strip("/").split("/")

    if len(parts) >= 2:
        username = parts[0]
        project_name = parts[1]
        result = f"{username}-{project_name}"
        return result

In [3]:
repo = "https://github.com/princ3kr/Notebook-LM-Mini"

filename = get_filename(repo)
dir = f"../temp/{filename}"

if os.path.isdir(dir):
    print("Directory already exists")

Directory already exists


In [4]:
import subprocess

if os.path.isdir(dir):
    print("Directory already exists")
else:
    try:
        subprocess.run(["git", "clone", repo, dir], check=True)
        print("Clone successful!")
    except subprocess.CalledProcessError as e:
        print(f"Git command failed with error: {e}")


Directory already exists


In [5]:
# tree = ast.parse(files["src\\services\\tutor_service.py"])
# print(ast.dump(tree, indent=2))

def parse_file(source_code):
    tree = ast.parse(source_code)
    import_modules, import_names, classes, functions = [], [], [], []
    
    for node in ast.walk(tree):
        if isinstance(node, ast.ImportFrom):
            import_modules.append(node.module)
            import_names.append([alias.name for alias in node.names])

        if isinstance(node, ast.ClassDef):
            classes.append({
                "name": node.name,
                "line_start": node.lineno,
                "line_end": node.end_lineno
            })
            
        if isinstance(node, ast.FunctionDef):
            functions.append({
                "name": node.name,
                "line_start": node.lineno,
                "line_end": node.end_lineno
            })
                
    return {"import_modules": import_modules, "import_names": import_names, "classes": classes, "functions": functions}

In [6]:
ignores = { ".git", ".gitignore", ".lock", ".venv", "__pycache__", "node_modules", ".vscode", "pyproject.toml", "__init__.py", ".python-version", "requirements.txt" }

def get_structure(directory: str):
    for root, dirs, files in os.walk(directory):
        dirs[:] = sorted([d for d in dirs if d not in ignores])

        rel_dir = os.path.relpath(root, directory)
        
        if rel_dir == ".":
            level = 0
            display_name = os.path.basename(os.path.abspath(directory))
        else:
            level = rel_dir.count(os.sep) + 1
            display_name = os.path.basename(root)

        indent = "│   " * (level - 1) if level > 0 else ""
        connector = "├── " if level > 0 else ""
        print(f"{indent}{connector}{display_name}/")

        file_indent = "│   " * level
        sorted_files = sorted(files)
        for i, name in enumerate(sorted_files):
            if name not in ignores or name == "README.md":
                file_connector = "└── " if (i == len(sorted_files) - 1 and not dirs) else "├── "
                print(f"{file_indent}{file_connector}{name}")

def get_files(directory: str):
    inventory = {}
    for root, dirs, files in os.walk(directory):
        rel_dir = os.path.relpath(root, directory)
        if rel_dir == ".":
            rel_dir = ""
        
        for name in files:
            if (name not in ignores) and (name == "README.md" or name.endswith(".py")):
                full_path = os.path.join(root, name)

                with open(full_path, "r", encoding="utf-8") as f:
                    content = f.read()

                    parsed_structure = {"import_modules": [], "import_names": [], "classes": [], "functions": []}
                    if name.endswith(".py"):
                        try:
                            parsed_structure = parse_file(content)
                        except SyntaxError:
                            pass

                    parsed_structure['content'] = content
                    
                    posix_path = os.path.join(rel_dir, name).replace("\\", "/")
                    inventory[posix_path] = parsed_structure

    return inventory


In [7]:
# get_structure(dir)
files = get_files(directory=dir)

In [8]:
for imports in files['src/database/neo4j_conn.py']:
    print(imports)

import_modules
import_names
classes
functions
content


In [9]:
import networkx as nx
from pyvis.network import Network
from neo4j import GraphDatabase
import json

class BuildGraph:
    def __init__(self, files: dict):
        self.G = nx.DiGraph()
        self.files = files
        self.name_index = self.build_name_index()
        self.module_index = self.build_module_index()

    def build_name_index(self) -> dict:
        name_index = {}
        for filepath in self.files:
            for classname in self.files.get(filepath, {}).get("classes", []):
                name_index[classname['name']] = filepath
            for funcname in self.files.get(filepath, {}).get("functions", []):
                name_index[funcname['name']] = filepath
        return name_index

    def build_module_index(self) -> dict:
        """Creates an index mapping python module paths (e.g. 'utils.helpers') to file paths"""
        module_index = {}
        for filepath in self.files:
            if filepath.endswith(".py"):
                # Convert 'src/utils/helpers.py' -> 'src.utils.helpers' and 'utils.helpers'
                parts = filepath[:-3].split("/")
                for i in range(len(parts)):
                    module_name = ".".join(parts[i:])
                    module_index[module_name] = filepath
        return module_index

    def get_path(self) -> dict:
        cleared_files = {}
        for filepath in self.files:
            imports = []
            modules = self.files[filepath].get("import_modules", [])
            names = self.files[filepath].get("import_names", [])

            for imp, imp_names in zip(modules, names):
                if not imp:
                    continue

                # Dynamically resolve import using the module index
                if imp in self.module_index:
                    imports.append(self.module_index[imp])
                else:
                    # Fallback to checking specific imported names (classes/functions)
                    for name in imp_names:
                        if name in self.name_index:
                            imports.append(self.name_index[name])
                            
            cleared_files[filepath] = list(set(imports)) # Remove duplicates
        return cleared_files

    def get_node_style(self, filepath):
        """Dynamically assigns colors based on top-level directory"""
        parts = filepath.split("/")
        top_level = parts[0] if len(parts) > 1 else "root"
        
        colors = [
            {"background": "#3b82f6", "border": "#60a5fa", "highlight": "#93c5fd"},
            {"background": "#10b981", "border": "#34d399", "highlight": "#6ee7b7"},
            {"background": "#f59e0b", "border": "#fbbf24", "highlight": "#fcd34d"},
            {"background": "#8b5cf6", "border": "#a78bfa", "highlight": "#c4b5fd"},
            {"background": "#ec4899", "border": "#f472b6", "highlight": "#fbcfe8"},
            {"background": "#ef4444", "border": "#f87171", "highlight": "#fca5a5"},
        ]
        
        color_idx = hash(top_level) % len(colors)
        return colors[color_idx]

    def build(self):
        cleared_path = self.get_path()
        for source, targets in cleared_path.items():
            style = self.get_node_style(source)
            self.G.add_node(
                source, 
                label=source.split("/")[-1],
                title=source,
                color=style,
                shadow=True,
                shape="dot",
                size=25,
                font={"color": "white", "size": 14, "face": "Segoe UI, Tahoma, Geneva, Verdana, sans-serif"}
            )
            for target in targets:
                self.G.add_edge(source, target)

    def push_to_neo4j(self, uri, user, password):
        driver = GraphDatabase.driver(uri, auth=(user, password))
        repo_id = filename

        with driver.session() as session:
            for node in self.G.nodes():
                imports = [
                    name for names in self.files[node]['import_names'] 
                    for name in names if name in self.name_index
                ]

                query = '''
                    MERGE (r:Repo {repo_id: $repo_id})
                    MERGE (f:File {path: $path, repo_id: $repo_id})
                    MERGE (r)-[:CONTAINS]->(f)
                    SET f.name = $name,
                        f.classes = $classes,
                        f.imports = $imports,
                        f.functions = $functions
                '''
                session.run(
                    query,
                    name=node.split('/')[-1],
                    path=node,
                    classes=[c['name'] for c in self.files.get(node, {}).get("classes", [])],
                    imports=imports,
                    functions=[f['name'] for f in self.files.get(node, {}).get("functions", [])],
                    repo_id=repo_id
                )
                
            for source, target in self.G.edges():
                query = '''
                    MATCH (a:File {path: $source, repo_id: $repo_id})
                    MATCH (b:File {path: $target, repo_id: $repo_id})
                    MERGE (a)-[:IMPORTS]->(b)
                '''
                session.run(query, source=source, target=target, repo_id=repo_id)
                
    def show(self, filename="graph.html"):
        net = Network(
            directed=True, notebook=True, cdn_resources='remote', 
            height="750px", width="100%", bgcolor="#111827", font_color="white"
        )
        net.from_nx(self.G)
        options = {
            "edges": {"color": {"inherit": "from", "opacity": 0.5}, "smooth": {"type": "continuous", "roundness": 0.5}, "arrows": {"to": {"enabled": True, "scaleFactor": 0.5}}, "width": 1.5},
            "physics": {"forceAtlas2Based": {"gravitationalConstant": -60, "centralGravity": 0.005, "springLength": 150, "springStrength": 0.08, "damping": 0.4, "avoidOverlap": 0.5}, "maxVelocity": 50, "minVelocity": 0.1, "solver": "forceAtlas2Based", "stabilization": {"enabled": True, "iterations": 1000}},
            "interaction": {"hover": True, "navigationButtons": True, "multiselect": True, "tooltipDelay": 200}
        }
        net.set_options(json.dumps(options))
        return net.write_html(filename)


In [10]:
builder = BuildGraph(files)
builder.build()
builder.show()

In [11]:
# builder.push_to_neo4j("bolt://localhost:7687", "neo4j", "pk142145")

In [ ]:
import os
import uuid
from dotenv import load_dotenv
from qdrant_client import QdrantClient

load_dotenv()

class VectorStore:
    def __init__(self, files: dict, collection_name: str = 'prototype'):
        self.files = files
        self.collection_name = collection_name
        self.chunks = []
        
        url = os.getenv("QDRANT_END_POINT")
        api_key = os.getenv("QDRANT_API_KEY")
        
        if not url or not api_key:
            raise ValueError("Please set QDRANT_END_POINT and QDRANT_API_KEY in your .env file!")
            
        self.client = QdrantClient(url=url, api_key=api_key)
        
        # We replace OpenAI with Qdrant's automatic dual-encoders
        self.client.set_model("BAAI/bge-small-en-v1.5")
        self.client.set_sparse_model("Qdrant/bm25")
        
        if not self.client.collection_exists(self.collection_name):
            self.client.create_collection(
                collection_name=self.collection_name,
                vectors_config=self.client.get_fastembed_vector_params(),
                sparse_vectors_config=self.client.get_fastembed_sparse_vector_params()
            )
            
        # Create a payload index on the "path" field so we can filter by it
        self.client.create_payload_index(
            collection_name=self.collection_name,
            field_name="path",
            field_schema="keyword"
        )


    def build(self, max_module_lines=100, overlap=5):
        self.chunks = []
        for file in self.files.keys():
            content = self.files[file]['content']
            lines = content.split('\n')
            classes = self.files[file]['classes']
            functions = self.files[file]['functions']

            if not functions:
                for i in range(0, len(lines), max_module_lines):
                    start = max(0, i - overlap)
                    end = min(len(lines), i + overlap + max_module_lines)
                    chunk_lines = lines[start:end]
                    self.chunks.append([
                        "\n".join(chunk_lines),
                        {
                            "path": file,
                            "filename": file.split("/")[-1],
                            "function_name": "module_level_segment",
                            "class_name": "module_level",
                            "line_start": start + 1,
                            "line_end": end
                        }
                    ])

            for func in functions:
                parent = 'module_level'
                for c in classes:
                    if func['line_start'] >= c['line_start'] and func['line_start'] <= c['line_end']:
                        parent = c['name']
                        break

                start = max(0, func['line_start'] - overlap)
                end = min(len(lines), func['line_end'] + overlap)
                chunk = "\n".join(lines[start:end])
                metadata = {
                    "path": file.replace("\\", "/"),
                    "filename": file.split("/")[-1],
                    "function_name": func['name'],
                    "class_name": parent,
                    "line_start": start + 1,
                    "line_end": end
                }
                self.chunks.append([chunk, metadata])

    def push(self):
        if not self.chunks:
            print("No chunks to push. Run build() first.")
            return

        documents = [c[0] for c in self.chunks]
        metadatas = [c[1] for c in self.chunks]

        ids = [
            str(uuid.uuid5(uuid.NAMESPACE_DNS, f"{m['path']}:{m['function_name']}:{i}")) 
            for i, m in enumerate(metadatas)
        ]

        self.client.add(
            collection_name=self.collection_name,
            documents=documents,
            metadata=metadatas,
            ids=ids
        )
        print(f"Successfully pushed {len(documents)} chunks to Qdrant Cloud!")

    def search(self, query: str, query_filter=None, top_k: int = 5):
        results = self.client.query(
            collection_name=self.collection_name,
            query_text=query,
            query_filter=query_filter,
            limit=top_k
        )

        return {
            "documents": [[hit.document for hit in results]],
            "metadatas": [[hit.metadata for hit in results]],
            "ids": [[str(hit.id) for hit in results]]
        }


p:\repo-2-graph\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
client = VectorStore(files, collection_name='R2Mapper')
client.build()

Fetching 18 files: 100%|██████████| 18/18 [00:01<00:00, 11.11it/s]


In [14]:
client.push()

p:\repo-2-graph\.venv\Lib\site-packages\qdrant_client\common\client_warnings.py:7: UserWarning: `add` method has been deprecated and will be removed in 1.17. Instead, inference can be done internally within regular methods like `upsert` by wrapping data into `models.Document` or `models.Image`.
  warnings.warn(message, category, stacklevel=stacklevel)


Successfully pushed 73 chunks to Qdrant Cloud!


In [15]:
from groq.types.chat import chat_completion_chunk
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import SystemMessage, HumanMessage
from pydantic import BaseModel, Field
from typing import Literal
from sentence_transformers import CrossEncoder
from fuzzywuzzy import process, fuzz
from qdrant_client.models import Filter, FieldCondition, MatchAny
import re
import os

groq_api_key = os.getenv('GROQ_API_KEY')

QUERY_TEMPLATES = {
    "leaf_nodes":     "MATCH (r:Repo {repo_id: $repo})-[:CONTAINS]->(f:File) WHERE NOT (f)-[:IMPORTS]->() RETURN f.path, null as imported_path",
    "direct_imports": "MATCH (r:Repo {repo_id: $repo})-[:CONTAINS]->(f:File {name: $name}) OPTIONAL MATCH (f)-[:IMPORTS]->(i) RETURN f.path, i.path as imported_path",
    "most_deps":      "MATCH (r:Repo {repo_id: $repo})-[:CONTAINS]->(f:File) OPTIONAL MATCH (f)-[:IMPORTS]->(i) RETURN f.path, COUNT(i) as dependencies_count, collect(i.path) as imported_paths ORDER BY dependencies_count DESC LIMIT 1",
    "indirect_deps":  "MATCH (r:Repo {repo_id: $repo})-[:CONTAINS]->(f:File {name: $source})-[:IMPORTS]->(mid:File {name: $via}) OPTIONAL MATCH (mid)-[:IMPORTS]->(i) RETURN f.path, mid.path as via_path, i.path as imported_path",
    "all_files":      "MATCH (r:Repo {repo_id: $repo})-[:CONTAINS]->(f:File) OPTIONAL MATCH (f)-[:IMPORTS]->(i) RETURN f.path, i.path as imported_path",
}

class QueryTemplate(BaseModel):
    template_name: Literal['leaf_nodes', 'direct_imports', 'most_deps', 'indirect_deps', 'all_files']
    params: dict = Field(default_factory=dict, description="Parameters to fill: name, source, via")

VALID_FILE_PROPERTIES = {'name', 'path', 'functions', 'classes', 'imports', 'repo_id'}

class CypherQuery(BaseModel):
    cypher: str = Field(..., description="Cypher query for retrieving relationships within the nodes")

class RouterDecision(BaseModel):
    decision: Literal['hybrid', 'graph_only'] = Field(
        ...,
        description="graph_only ONLY for pure structural/dependency/topology questions. hybrid for everything else."
    )

class QueryEngine:
    def __init__(self, repo_id: str, db_client, uri: str, user: str, password: str, llm: ChatGroq):
        self.db_client = db_client
        self.repo_id = repo_id
        self.llm = llm
        self.graph_driver = GraphDatabase.driver(uri, auth=(user, password))
        self.rerank_model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
        self._file_index = None
        self._router_cache: dict[str, RouterDecision] = {}
        self._graph_cache: dict[str, dict | None] = {}


    def _load_file_index(self) -> list[dict]:
        """
        Fetch all files with their classes and functions from Neo4j.
        Cached after first call so we don't hit the DB on every query.
        """
        if self._file_index is not None:
            return self._file_index

        with self.graph_driver.session() as session:
            result = session.run(f"""
                MATCH (r:Repo {{repo_id: '{self.repo_id}'}})-[:CONTAINS]->(f:File)
                RETURN f.name as name, f.path as path,
                       f.classes as classes, f.functions as functions
            """)
            self._file_index = [dict(r) for r in result]

        return self._file_index

    def resolve_filename(self, raw_name: str) -> str | None:
        """
        Fuzzy-match a raw name (possibly missing extension or slightly misspelled)
        against actual filenames in the graph. Returns the matched name or None.
        """
        index = self._load_file_index()
        names = [f["name"] for f in index]

        if raw_name in names:
            return raw_name

        match = process.extractOne(raw_name, names, scorer=fuzz.token_sort_ratio)
        if match and match[1] >= 70:
            return match[0]

        return None

    def extract_entities_from_query(self, query: str) -> list[str]:
        """
        Pull class/function/file mentions from the query text and map them
        to file paths via fuzzy matching. Used when graph returns no filenames.
        """
        index = self._load_file_index()

        entity_map: dict[str, str] = {}
        for f in index:
            entity_map[f["name"]] = f["path"]
            for cls in (f.get("classes") or []):
                entity_map[cls] = f["path"]
            for fn in (f.get("functions") or []):
                entity_map[fn] = f["path"]

        if not entity_map:
            return []

        words = query.replace("'s", "").split()
        bigrams = [f"{words[i]} {words[i+1]}" for i in range(len(words) - 1)]
        candidates = words + bigrams

        matched_paths = []
        for candidate in candidates:
            match = process.extractOne(
                candidate,
                list(entity_map.keys()),
                scorer=fuzz.token_sort_ratio
            )
            if match and match[1] >= 75:
                matched_paths.append(entity_map[match[0]])

        return list(set(matched_paths))

    def sanitize_cypher(self, cypher: str) -> str:
        # Catch any property access: .name, .path, etc.
        used_props = set(re.findall(r'\.(\w+)', cypher))
        invalid = used_props - VALID_FILE_PROPERTIES
        if invalid:
            # We log but return None to trigger the safe entity-extraction fallback
            print(f"Sanitization blocked invalid properties: {invalid}")
            return None
        return cypher

    def resolve_names_in_cypher(self, cypher: str) -> str:
        """
        Find name/source/via string literals in generated Cypher and
        fuzzy-resolve them to actual filenames in the graph.
        """
        def replacer(match):
            raw = match.group(1)
            resolved = self.resolve_filename(raw)
            if resolved and resolved != raw:
                print(f"[resolve] '{raw}' → '{resolved}'")
                return match.group(0).replace(raw, resolved)
            return match.group(0)

        pattern = r"\{(?:name|source|via):\s*['\"]([^'\"]+)['\"]\}"
        return re.sub(pattern, replacer, cypher)

    def _is_meaningful(self, graph_result: dict) -> bool:
        """
        A graph result is meaningful if it exists, is not a sanitization failure (None),
        and is not the LLM's fallback sentinel response.
        """
        if graph_result is None:
            return False
        return not graph_result.get("is_fallback", False) and bool(graph_result.get("data"))


    def graph_search(self, query: str) -> dict | None:
        if query in self._graph_cache:
            return self._graph_cache[query]
        structured_llm = self.llm.with_structured_output(CypherQuery)
        
        index = self._load_file_index()
        sample = index[:10]
        file_list = "\n".join(f"- {f['name']} ({f['path']})" for f in sample)

        system_content = f"""
            You are a Cypher expert for Neo4j. Generate ONLY valid queries for this schema:
            Nodes: Repo(repo_id), File(name, path, functions, classes, imports)
            Edges: (Repo)-[:CONTAINS]->(File), (File)-[:IMPORTS]->(File)
            VALID PROPERTIES: {', '.join(VALID_FILE_PROPERTIES)}
            
            RESOURCES IN THIS REPO:
            {file_list}
            RULES:
            1. NEVER use properties from the question (like 'planned_paths' or 'status') if they aren't in the VALID list.
            2. For behavioral questions (how/why/what happens), return this sentinel:
            MATCH (r:Repo {{repo_id: '{self.repo_id}'}})-[:CONTAINS]->(f:File) RETURN f.path LIMIT 1
            3. Always start with: MATCH (r:Repo {{repo_id: '{self.repo_id}'}})
        """
        response = structured_llm.invoke([
            SystemMessage(content=system_content),
            HumanMessage(content=query)
        ])
        try:
            safe_cypher = self.sanitize_cypher(response.cypher)
            if not safe_cypher: return None
            
            safe_cypher = self.resolve_names_in_cypher(safe_cypher)
            with self.graph_driver.session() as session:
                result = session.run(safe_cypher)
                data = [record.data() for record in result]
            is_fallback = (len(data) == 1 and len(data[0]) == 1 and "f.path" in data[0])
            output = {"is_fallback": is_fallback, "data": data}
            self._graph_cache[query] = output 
            return output
        except Exception as e:
            return None


    def vector_search(self, query: str, filenames: list, top_k: int = 5):
        qdrant_filter = Filter(
            must=[
                FieldCondition(
                    key="path",
                    match=MatchAny(any=filenames)
                )
            ]
        )
        
        return self.db_client.search(
            query=query, 
            query_filter=qdrant_filter, 
            top_k=top_k
        )

        

    def rerank(self, context: dict, query: str, top_k: int):
        documents = context['documents'][0]
        metadatas = context['metadatas'][0]

        chunks = [(query, chunk) for chunk in documents]
        scores = self.rerank_model.predict(chunks)

        return sorted(
            [[metadatas[i], score, documents[i]] for i, score in enumerate(scores)],
            key=lambda x: x[1],
            reverse=True
        )[:top_k]

    def _extract_filenames(self, graph_data: list[dict]) -> list[str]:
        """
        Pull every non-null file path out of a graph result list.
        Uses suffix heuristics instead of a hardcoded key whitelist so it
        handles any Cypher alias the LLM might generate:
        - keys ending in '.path'  → single path string  (f.path, upstream.path, imported.path, via_path, etc.)
        - keys ending in '_paths' → list of path strings (imported_paths, etc.)
        - keys ending in '_chain' → list of path strings (dependency_chain, etc.)
        """
        paths = []
        for record in graph_data:
            for k, v in record.items():
                if v is None:
                    continue

                # Single path string: any key whose name ends with '.path' or '_path'
                if isinstance(v, str) and (k.endswith(".path") or k.endswith("_path")):
                    paths.append(v)

                # List of paths: any key whose name ends with '_paths' or '_chain'
                elif isinstance(v, list) and (k.endswith("_paths") or k.endswith("_chain")):
                    paths.extend(p for p in v if isinstance(p, str) and p)

        return list(set(paths))


    def _resolve_filenames(self, graph_result: dict | None, query: str) -> list[str]:
        """
        Try to get filenames from graph result first.
        If graph result is None, fallback, or empty — extract entities from query text.
        """
        if graph_result and self._is_meaningful(graph_result):
            filenames = self._extract_filenames(graph_result["data"])
            if filenames:
                return filenames

        print("[fallback] Graph returned no meaningful data, using entity extraction")
        return self.extract_entities_from_query(query)


    def router(self, query: str) -> RouterDecision:
        if query in self._router_cache:
            return self._router_cache[query]

        router_llm = self.llm.with_structured_output(RouterDecision)
        router_prompt = ChatPromptTemplate.from_messages([
            ('system', """
                You are a query router for a code repository Q&A assistant.

                The knowledge graph stores ONLY file-level structural data:
                which files exist, and which file imports which other file.
                It has NO knowledge of function behavior, variable values,
                runtime state, class internals, or code logic.

                Classify the query into exactly one of:

                graph_only — answerable purely from import structure:
                - which files import X
                - what does <file> depend on
                - which files have no imports
                - which file has the most dependencies
                - transitive dependencies of <file>
                - what files depend on <file> (reverse lookup)

                hybrid — everything else:
                - what a function/method does or returns
                - what happens when a condition is met
                - initial / default value of anything
                - how a feature is implemented
                - what database / framework / library is used
                - any question about runtime behavior, state, or logic
                - any question containing: "what happens if", "how does",
                    "what does X do", "what is the value of", "why does"

                RULE: if the question mentions a specific variable or field name,
                or asks about behavior — always choose hybrid.
                When in doubt, always choose hybrid.
            """),
            ('user', "query: {query}")
        ])


        result = (router_prompt | router_llm).invoke({"query": query})
        self._router_cache[query] = result
        return result


    def get_result(self, query: str, top_k: int):
        route = self.router(query).decision

        def _pure_vector():
            vector_data = self.db_client.search(query=query, top_k=top_k)

            return {
                "type": "hybrid",
                "graph": [],
                "vector": self.rerank(vector_data, query, top_k),
            }

        if route == "graph_only":
            graph_result = self.graph_search(query)

            if graph_result is None:
                return _pure_vector()

            if self._is_meaningful(graph_result):
                return {
                    "type": "graph",
                    "graph": graph_result["data"],
                }

            filenames = self.extract_entities_from_query(query)

            if filenames:
                vector_data = self.vector_search(query, filenames=filenames)
            else:
                vector_data = self.db_client.search(query=query, top_k=top_k)

            return {
                "type": "hybrid",
                "graph": [],
                "vector": self.rerank(vector_data, query, top_k),
            }

        else:
            graph_result = self.graph_search(query)

            if graph_result is None:
                return _pure_vector()

            filenames = self._resolve_filenames(graph_result, query)
            
            if filenames:
                vector_data = self.vector_search(query, filenames=filenames, top_k=top_k)
            else:
                vector_data = self.db_client.search(query=query, top_k=top_k)
            return {
                "type": "hybrid",
                "graph": graph_result["data"],
                "vector": self.rerank(vector_data, query, top_k),
            }


    def close(self):
        self.graph_driver.close()

p:\repo-2-graph\.venv\Lib\site-packages\fuzzywuzzy\fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


In [16]:
from langchain_openai import ChatOpenAI

routing_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

engine = QueryEngine(
    repo_id=filename,
    db_client=client,
    uri="bolt://localhost:7687",
    user="neo4j",
    password="pk142145",
    llm=routing_llm
)

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 4052.80it/s]


In [17]:
def format_documents(data: dict) -> str:
    context = ""

    graph_records = data.get("graph", [])
    if graph_records:
        context += "[Graph Relationships]\n"
        for r in graph_records:
            for k, v in r.items():
                if v is None: continue
                if isinstance(v, list):
                    context += f"  {k}: {', '.join(str(x) for x in v if x)}\n"
                else:
                    context += f"  {k}: {v}\n"
            context += "\n"

    if data.get('type') == 'hybrid':
        vector = data.get("vector", [])
        if vector:
            context += "[Code Chunks]\n"
            for meta, _, doc in vector:
                context += f"\n[{meta.get('path', 'unknown')} | lines {meta.get('line_start', '?')}-{meta.get('line_end', '?')}]\n{doc}\n"

    return context.strip()


In [19]:
query = "which is the constructor of IntentServices is using?"
result = engine.get_result(query, 6)
formated_result = format_documents(result)

Sanitization blocked invalid properties: {'py'}


p:\repo-2-graph\.venv\Lib\site-packages\qdrant_client\common\client_warnings.py:7: UserWarning: `query` method has been deprecated and will be removed in 1.17. Instead, inference can be done internally within regular methods like `query_points` by wrapping data into `models.Document` or `models.Image`.
  warnings.warn(message, category, stacklevel=stacklevel)


In [20]:
from pydantic import BaseModel, Field

class ResponseModel(BaseModel):
    answer: str = Field(..., description="Response of the query from the llm model")
    score: float = Field(..., description="Confidence score of the response")
    sources: str = Field(..., description="File name used for response")

class AnswerEngine:
    def __init__(self, llm):
        self.llm = llm.with_structured_output(ResponseModel)

    def generate_response(self, query: str, context: str):
        prompt = ChatPromptTemplate.from_messages([
            ("system", """
                You are a code repository assistant.
                Answer only from the provided context.
                When asked about initial or default values, prioritize code that
                constructs or initializes objects (e.g. GraphState(...), __init__,
                initial_state) over code that updates or transitions values.
                Sources should be file names used to answer.
                Confidence should be between 0 and 1.
            """),
            ("user", "Context:\n{context}\n\nQuestion: {query}")
        ])
        chain = prompt | self.llm
        return chain.invoke({"context": context, "query": query})

In [21]:
from langchain_openai import ChatOpenAI

answer_llm = ChatOpenAI(model="gpt-4o", temperature=0)
chat_engine = AnswerEngine(answer_llm)
result = chat_engine.generate_response(query, formated_result)

In [22]:
result.answer

"The constructor of `IntentService` is defined in the `src/services/intent_service.py` file. It is as follows:\n\n```python\nclass IntentService:\n    def __init__(self, neo4j_conn):\n        self.driver = neo4j_conn.connect()\n        self.biencoder = SentenceTransformer('all-MiniLM-L6-v2')\n        self.crossencoder = CrossEncoder('cross-encoder/stsb-roberta-base')\n        self.topics, self.topic_embeddings = self._load_node_embeddings()\n```\n\nThis constructor initializes the `IntentService` with a Neo4j connection, a biencoder, a crossencoder, and loads node embeddings."

## RAG Evaluation metrices

In [23]:
eval_questions = [

    # --- EDGE: FILES WITH NO IMPORTS ---
    {
        "question": "Which files in the repository do not import any other file?",
        "ground_truth": "Files with no imports are leaf nodes in the dependency graph — likely src/models/concept.py, src/models/state.py, src/models/diagnoses.py, and src/models/intent.py as they are pure model definitions"
    },

    # --- EDGE: CIRCULAR / DEEP DEPENDENCY ---
    {
        "question": "Which service has the most dependencies on other files?",
        "ground_truth": "src/services/tutor_service.py has the most dependencies as it imports Neo4jConn, StudentDB, DiagnoserService, IntentService, PlannerService, and tutor_lesson_utils"
    },

    # --- EDGE: SPECIFIC FUNCTION BEHAVIOR ---
    {
        "question": "What happens if planned_paths is empty in the planner node?",
        "ground_truth": "If planned_paths is empty, the planner node returns Command with goto=END and a final_response message explaining either the topic was not recognized or no learning path was found"
    },

    # --- EDGE: SPECIFIC CLASS METHOD ---
    {
        "question": "What does update_last_json_path do in StudentDB and when is it called?",
        "ground_truth": "update_last_json_path updates the last_json_path field and last_updated timestamp in TinyDB for the student. It is called in app.py after successfully building the knowledge graph from a PDF"
    },

    # --- EDGE: STATE FIELD USAGE ---
    {
        "question": "What is the initial value of phase in GraphState when a new session starts?",
        "ground_truth": "The initial value of phase is 'quiz' as set in the initial_state construction in app.py"
    },

    # --- EDGE: ERROR PATH ---
    {
        "question": "What does app.py do if no concepts are extracted from the uploaded PDF?",
        "ground_truth": "app.py updates status to error state, displays an error message saying no relevant concepts were identified and to try a more detailed document, then calls st.stop()"
    },

    # --- EDGE: MULTI-HOP DEPENDENCY ---
    {
        "question": "Which files does app.py indirectly depend on through tutor_service.py?",
        "ground_truth": "Through tutor_service.py, app.py indirectly depends on neo4j_conn.py, student_db.py, diagnoser_service.py, intent_service.py, planner_service.py, and tutor_lesson_utils.py"
    },

    # --- EDGE: SPECIFIC PARAMETER ---
    {
        "question": "What is the process_limit in app.py and what does it control?",
        "ground_truth": "process_limit is set to 50 in app.py and controls the maximum number of PDF chunks processed for concept extraction to avoid excessive API calls"
    },

    # --- EDGE: TWO LLM INSTANCES ---
    {
        "question": "Why does TutorWorkflow initialize two separate LLM instances?",
        "ground_truth": "TutorWorkflow uses self.llm with max_tokens=250 for quick responses like quiz questions and self.teach_llm with max_tokens=1000 and temperature=0.35 for longer concept explanations requiring more detail"
    },

    # --- EDGE: CHECKPOINT ---
    {
        "question": "What checkpointing mechanism does TutorWorkflow use and why?",
        "ground_truth": "TutorWorkflow uses MemorySaver from langgraph.checkpoint.memory as the checkpointer when compiling the workflow, which enables conversation state persistence across multiple turns using thread_id"
    },

    # --- EDGE: THREAD ID ---
    {
        "question": "How is thread_id constructed in app.py and what is it used for?",
        "ground_truth": "thread_id is constructed as f'thread_{student_id}' in app.py and is used as the LangGraph config key to isolate each student's conversation state in MemorySaver"
    },

    # --- EDGE: RESUME VS FRESH ---
    {
        "question": "What condition determines whether the tutor resumes an existing session or starts fresh?",
        "ground_truth": "If state.values and state.next both exist, the tutor resumes via Command(resume=user_input). If either is falsy, it starts fresh by invoking with a new GraphState"
    },

    # --- EDGE: TEMP FILE CLEANUP ---
    {
        "question": "How does app.py handle the temporary PDF file after processing?",
        "ground_truth": "After processing, app.py checks if the temp file exists using os.path.exists and removes it using os.remove to clean up disk space"
    },

    # --- EDGE: MISSING INPUT GUARD ---
    {
        "question": "How does app.py handle the case where user types the placeholder text 'I want to learn about...' literally?",
        "ground_truth": "app.py checks if user_input.strip().lower() equals 'i want to learn about...' or 'i want to learn about' and replaces user_input with an empty string to avoid sending the placeholder as actual input"
    },

    # --- EDGE: DATABASE LAYER ---
    {
        "question": "What database does StudentDB use internally and what file does it store data in?",
        "ground_truth": "StudentDB uses TinyDB internally and stores data in a JSON file at the path db_folder/student_sessions.json where db_folder is determined by the student_id"
    }
]

In [24]:
results = []

for sample in eval_questions:
    raw_result = engine.get_result(sample["question"], top_k=5)
    context_text = format_documents(raw_result)
    response = chat_engine.generate_response(sample["question"], context_text)
   
    results.append({
        "question": sample["question"],
        "answer": response.answer,
        "contexts": [context_text],
        "ground_truth": sample["ground_truth"]
    })


[fallback] Graph returned no meaningful data, using entity extraction


UnexpectedResponse: Unexpected Response: 400 (Bad Request)
Raw response content:
b'{"status":{"error":"Bad request: Index required but not found for \\"path\\" of one of the following types: [keyword]. Help: Create an index for this key or use a different filter."},"time":0.000027265}'

In [40]:
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    answer_correctness,
    context_precision,
    context_recall
)
from ragas.llms import LangchainLLMWrapper
from datasets import Dataset

ragas_llm = LangchainLLMWrapper(ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
))

dataset = Dataset.from_list(results)

scores = evaluate(
    dataset,
    metrics=[
        faithfulness,
        answer_relevancy,
        answer_correctness,
        context_precision,
        context_recall
    ],
    llm=ragas_llm
)

print(scores)


Evaluating: 100%|██████████| 75/75 [00:48<00:00,  1.55it/s]

{'faithfulness': 0.7567, 'answer_relevancy': 0.8676, 'answer_correctness': 0.5991, 'context_precision': 0.9333, 'context_recall': 0.6111}
